# LatentMind V6 — Server Notebook

**Purpose:** load the models once, then serve the Djezzy AI API behind an ngrok tunnel.  
No training, no benchmarks — just warm-up and serve.

| Step | Cell | Notes |
|------|------|-------|
| 1 | GPU cleanup | run before any restart |
| 2 | Mount Drive + pull repo | clones on first run, pulls on subsequent runs |
| 3 | Install deps | one-shot: inference + voice + server |
| 4 | Configure env | model size, paths, tokens |
| 5 | Database | copy SQLite from Drive |
| 6 | Train brain | one-time MLP training (~2 min); skips if cached on Drive |
| 7 | Load brain + SLM | Qwen 4B + polisher + BGE-M3 |
| 8 | Load voice models | faster-whisper STT + XTTS-v2 TTS |
| 9 | **Serve** | start FastAPI + ngrok — **leave running** |

> **Re-serve after a code change:** just rerun cell 8 — it kills the old process and reloads `v6.server` from disk automatically. No model reload needed.

In [2]:
# ── CELL 1: GPU cleanup ───────────────────────────────────────────────────────
# Run this before restarting any model cell to avoid OOM.
import gc, sys, torch

def cleanup(verbose: bool = True, _globals: dict | None = None) -> None:
    freed: list[str] = []
    _g = _globals or {}
    for name in ('agent', 'slm', 'stt', 'tts'):
        if name in _g:
            del _g[name]
            freed.append(f'global:{name}')
    try:
        import v6.slm as _m
        if _m._slm is not None:
            for tid in list(_m._slm._store.keys()):
                _m._slm.clear_thread(tid)
            if hasattr(_m._slm, '_draft') and _m._slm._draft is not None:
                del _m._slm._draft
            del _m._slm._model, _m._slm._tok
            _m._slm = None
            freed.append('v6.slm')
    except Exception:
        pass
    try:
        import v6.speech as _s
        if getattr(_s, '_stt', None): _s._stt = None; freed.append('v6.stt')
        if getattr(_s, '_tts', None): _s._tts = None; freed.append('v6.tts')
    except Exception:
        pass
    try:
        import v6.brain as _b
        if getattr(_b, '_brain', None): _b._brain = None; freed.append('v6.brain')
    except Exception:
        pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    if verbose:
        print('Freed:', freed or 'nothing (first run)')
        if torch.cuda.is_available():
            a = torch.cuda.memory_allocated() / 1e9
            t = torch.cuda.get_device_properties(0).total_memory / 1e9
            print(f'GPU after cleanup: {a:.1f} GB / {t:.1f} GB')

print('cleanup() defined — call cleanup(verbose=True, _globals=globals()) to free VRAM')

cleanup() defined — call cleanup(verbose=True, _globals=globals()) to free VRAM


In [4]:
# ── CELL 2: Mount Drive + clone / pull repo ───────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, sys, shutil

REPO_URL = 'https://github.com/Hamza09Hamza/Latent-Djezzy.git'
REPO_DIR = '/content/Latent-Djezzy'
BRANCH   = 'main'

os.chdir('/content')
if os.path.isdir(os.path.join(REPO_DIR, '.git')):
    print(f'Repo found — pulling latest from origin/{BRANCH} ...')
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', '-B', BRANCH,
                    f'origin/{BRANCH}'], check=True)
    print(f'✓ repo updated')
else:
    if os.path.isdir(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    subprocess.run(['git', 'clone', '--depth=1', '--branch', BRANCH,
                    REPO_URL, REPO_DIR], check=True)
    print(f'✓ cloned {BRANCH} → {REPO_DIR}')

# Flush any cached v6 modules so the freshly pulled code is active
for _mod in list(sys.modules.keys()):
    if _mod.startswith('v6'):
        del sys.modules[_mod]

sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

commit = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'],
                                  cwd=REPO_DIR).decode().strip()
print(f'Commit: {commit}  |  Working dir: {os.getcwd()}')

Mounted at /content/drive
✓ cloned main → /content/Latent-Djezzy
Commit: 6ce7385  |  Working dir: /content/Latent-Djezzy


In [5]:
# ── CELL 3: Install all dependencies (one shot) ───────────────────────────────
# Core inference + RAG
!pip install -q 'transformers>=4.46.0' 'sentence-transformers>=3.0.0' 'accelerate>=0.27.0'
!pip install -q 'langgraph>=0.2.0' 'bitsandbytes>=0.43.0' scipy matplotlib
!pip install -q jinja2 pydantic pymysql mysql-connector-python
print('✓ inference + RAG deps')

# Voice layer — STT + TTS
!pip install -q faster-whisper soundfile rapidfuzz
!pip install -q coqui-tts
!pip install -q 'transformers>=4.46.0'   # re-assert after coqui may downgrade
print('✓ voice layer (STT + TTS)')

# API server + ngrok tunnel
!pip install -q fastapi uvicorn python-multipart pyngrok nest-asyncio
print('✓ server + tunnel deps')

print('\n✓ All dependencies installed')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 46.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.7/45.7 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 120.2 MB/s eta 0:00:0000:0100:01
✓ inference + RAG deps
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 76.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 15.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 76.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 70.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 126.9 MB/s eta 0:00:0000:010:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 862.8/862.8 kB 61.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 7.0 MB/s eta 0:00:00
   ━━━━━━

In [6]:
# ── CELL 4: Environment configuration ────────────────────────────────────────
import os, warnings, logging, datetime

warnings.filterwarnings('ignore')
logging.getLogger('transformers').setLevel(logging.ERROR)
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'

# Database
os.environ['V6_USE_SQLITE']  = '1'
os.environ['V6_SQLITE_PATH'] = '/content/interndb.sqlite'

# ── Model sizes ───────────────────────────────────────────────────────────────
# '4b' → Qwen3-4B  (~8 GB fp16)  ← default, fits comfortably on T4
# '3b' → Qwen2.5-Coder-3B        (~6.4 GB fp16)  — lighter option
# '7b' → Qwen2.5-Coder-7B        (~14 GB) — requires V6_4BIT=1
os.environ['V6_SLM_SIZE']        = '4b'
os.environ['V6_4BIT']            = '0'
os.environ['V6_SPECULATIVE']     = '1'   # 0.6B drafter → 2-4x speed
os.environ['V6_CONSTRAINED_SQL'] = '0'
os.environ['V6_POLISHER_HUB_ID'] = 'Qwen/Qwen2.5-1.5B-Instruct'
os.environ['V6_SLM_OVERRIDE']    = ''

# Output dirs on Drive
os.environ['V6_OUTPUT_DIR'] = '/content/drive/MyDrive/LatentDjezzy/v6_output'

# Memory allocator — prevents fragmentation on T4
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

# Date anchor — the model needs to know today's date for relative queries
os.environ['V6_DATE_ANCHOR'] = datetime.date.today().isoformat()

print('✓ environment configured')
print(f'  SLM size   : {os.environ["V6_SLM_SIZE"]}  (4bit={os.environ["V6_4BIT"]})')
print(f'  Date anchor: {os.environ["V6_DATE_ANCHOR"]}')
print(f'  DB path    : {os.environ["V6_SQLITE_PATH"]}')
print(f'  Output dir : {os.environ["V6_OUTPUT_DIR"]}')

✓ environment configured
  SLM size   : 4b  (4bit=0)
  Date anchor: 2026-05-31
  DB path    : /content/interndb.sqlite
  Output dir : /content/drive/MyDrive/LatentDjezzy/v6_output


In [7]:
# ── CELL 5: Database + output directories ────────────────────────────────────
import shutil, os

LOCAL_DB = '/content/interndb.sqlite'
possible_locations = [
    '/content/drive/MyDrive/LatentDjezzy/interndb.sqlite',
    '/content/drive/MyDrive/interndb.sqlite',
]
DRIVE_DB = next((p for p in possible_locations if os.path.isfile(p)), None)

if not DRIVE_DB:
    print('⚠  Database not found in Drive. Checked:')
    for p in possible_locations:
        print(f'    {p}')
else:
    if not os.path.isfile(LOCAL_DB):
        shutil.copy(DRIVE_DB, LOCAL_DB)
        print(f'✓ copied interndb.sqlite ({os.path.getsize(LOCAL_DB):,} bytes) → /content/')
    else:
        print(f'✓ SQLite already present: {LOCAL_DB}')

output_base = '/content/drive/MyDrive/LatentDjezzy/v6_output'
for d in [f'{output_base}/charts', f'{output_base}/emails',
          f'{output_base}/reports', f'{output_base}/audio']:
    os.makedirs(d, exist_ok=True)
print(f'✓ output dirs ready: {output_base}')

✓ copied interndb.sqlite (40,271,872 bytes) → /content/
✓ output dirs ready: /content/drive/MyDrive/LatentDjezzy/v6_output


In [ ]:
# ── CELL 6: Train brain MLP (one-time, ~2 min on T4) ─────────────────────────
# The brain is a tiny 3-head MLP (~200 KB).  It must be trained before
# loading models.  This cell is idempotent: it skips training if the file
# already exists (either from a previous run or restored from Drive).
import os, shutil, sys

BRAIN_PT  = '/content/Latent-Djezzy/models/brain_head.pt'
DRIVE_PT  = '/content/drive/MyDrive/LatentDjezzy/models/brain_head.pt'

os.makedirs(os.path.dirname(BRAIN_PT), exist_ok=True)

if os.path.isfile(BRAIN_PT):
    print(f'✓ brain_head.pt already present ({os.path.getsize(BRAIN_PT):,} bytes) — skipping training')

elif os.path.isfile(DRIVE_PT):
    shutil.copy(DRIVE_PT, BRAIN_PT)
    print(f'✓ brain_head.pt restored from Drive ({os.path.getsize(BRAIN_PT):,} bytes)')

else:
    print('brain_head.pt not found — training from scratch (~2 min on T4)...')
    print('Step 1/2: synthesizing agentic traces...')
    import subprocess
    r1 = subprocess.run(
        [sys.executable, '-m', 'v6.brain_data'],
        cwd='/content/Latent-Djezzy', capture_output=True, text=True
    )
    if r1.returncode != 0:
        print('STDERR:', r1.stderr[-2000:])
        raise RuntimeError('brain_data failed')
    print(r1.stdout.strip() or '  traces written')

    print('Step 2/2: training brain head...')
    r2 = subprocess.run(
        [sys.executable, '-m', 'v6.train_brain'],
        cwd='/content/Latent-Djezzy', capture_output=True, text=True
    )
    if r2.returncode != 0:
        print('STDERR:', r2.stderr[-2000:])
        raise RuntimeError('train_brain failed')
    print(r2.stdout.strip())

    # Back up to Drive so the next session skips training
    os.makedirs(os.path.dirname(DRIVE_PT), exist_ok=True)
    shutil.copy(BRAIN_PT, DRIVE_PT)
    print(f'✓ brain_head.pt saved to Drive for future sessions')


In [8]:
# ── CELL 7: Load brain + SLM + polisher ─────────────────────────────────────
# Run cleanup() first if you're reloading after an OOM.
import sys, os, torch
sys.path.insert(0, '/content/Latent-Djezzy')

try:
    cleanup(verbose=False, _globals=globals())
except NameError:
    pass  # first run — nothing to clean

from v6.graph import LatentMindV6
from v6.slm import get_slm, get_polisher
from v6.brain import get_brain

print('Loading BGE-M3 encoder + SLM (Qwen 4B) — first run downloads ~8 GB...')
agent = LatentMindV6()
get_slm()
get_brain()

print('Loading polisher (Qwen 1.5B)...')
try:
    get_polisher()
    print('✓ Polisher ready')
except Exception as _e:
    print(f'  Polisher unavailable ({_e}) — raw answers used instead')

if torch.cuda.is_available():
    alloc = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'\nGPU: {alloc:.1f} GB / {total:.1f} GB  (headroom ~{total - alloc:.1f} GB)')

Loading BGE-M3 encoder + SLM (Qwen 4B) — first run downloads ~8 GB...


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

FileNotFoundError: brain head not found at /content/Latent-Djezzy/models/brain_head.pt
The brain is a trained MLP — build it once before use:
    python3 -m v6.brain_data    # synthesize agentic traces
    python3 -m v6.train_brain   # train the head

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

In [ ]:
# ── CELL 8: Load voice models (STT + TTS) ────────────────────────────────────
# STT: faster-whisper large-v3  (transcribes the user's spoken question)
# TTS: XTTS-v2 cloned from your reference recordings in v6/audio/
#      French question → French voice | English question → English voice
#      Streaming: speech starts ~1 sentence after generation begins.
import os, torch
from v6.config import V6Config
from v6.speech import get_stt, get_tts

# On a 16 GB T4, uncomment to save ~1.5 GB on the STT model:
# os.environ['V6_STT_COMPUTE'] = 'int8_float16'

print('Loading STT (faster-whisper large-v3)...')
stt = get_stt()
print('Loading TTS (XTTS-v2)...')
tts = get_tts()

print('\n✓ Voice models ready — reference voices:')
for lang, label in (('fr', 'French'), ('en', 'English')):
    wav = V6Config.speaker_wav(lang)
    if wav and os.path.isfile(wav):
        print(f'  {label:8} → cloning  {os.path.basename(wav)}'
              f'  ({os.path.getsize(wav)//1024} KB)')
    else:
        name = V6Config.TTS_SPEAKER_FR if lang == 'fr' else V6Config.TTS_SPEAKER_EN
        print(f'  {label:8} → built-in studio voice: {name!r}')

if torch.cuda.is_available():
    a = torch.cuda.memory_allocated() / 1e9
    t = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'\nGPU: {a:.1f} GB / {t:.1f} GB  (headroom ~{t - a:.1f} GB)')

In [ ]:
# ── CELL 9: SERVE ─────────────────────────────────────────────────────────────
# Starts FastAPI + ngrok.  Models stay in VRAM (no reload).
# Safe to rerun: kills the old uvicorn process and reloads v6.server from disk.
# ─────────────────────────────────────────────────────────────────────────────
import os, sys, time, threading, nest_asyncio, uvicorn
from pyngrok import ngrok

# ── Config ────────────────────────────────────────────────────────────────────
NGROK_TOKEN = '3ERc2GB0MMOjOt2dleZiNurkxZl_55C3LUxsrrK7hH8qFtBx7'  # ← your token
os.environ['V6_API_TOKEN'] = os.environ.get('V6_API_TOKEN') or 'djezzy-demo'
assert NGROK_TOKEN, 'Paste your ngrok authtoken into NGROK_TOKEN above.'

# ── 1. Kill any stale uvicorn on port 8000 ────────────────────────────────────
os.system('fuser -k 8000/tcp 2>/dev/null')
time.sleep(2)

# ── 2. Flush v6 module cache so git pull changes are always picked up ─────────
#    (models stay loaded — only v6.server + its imports are reloaded)
for _mod in list(sys.modules.keys()):
    if _mod.startswith('v6'):
        del sys.modules[_mod]

# ── 3. ngrok tunnel ───────────────────────────────────────────────────────────
nest_asyncio.apply()
ngrok.set_auth_token(NGROK_TOKEN)
for _t in ngrok.get_tunnels():
    ngrok.disconnect(_t.public_url)

# ── 4. Import server (fresh from disk) ───────────────────────────────────────
from v6.server import app, API_TOKEN

# ── 5. Start uvicorn in a background thread ───────────────────────────────────
#    Runs inside this Python process → models already in VRAM, zero reload time.
_cfg = uvicorn.Config(app, host='0.0.0.0', port=8000, log_level='warning')
_srv = uvicorn.Server(_cfg)
_srv.install_signal_handlers = lambda: None
threading.Thread(target=_srv.run, daemon=True).start()
time.sleep(4)

# ── 6. Open ngrok tunnel ──────────────────────────────────────────────────────
_public = ngrok.connect(8000, 'http').public_url

print('[server] models warm — ready')
print('=' * 66)
print('Web UI    :', _public, ' (open in a browser)')
print('API token :', API_TOKEN, ' (paste it in the Settings panel)')
print('WebSocket :', _public.replace('https', 'wss') + '/ws?token=' + API_TOKEN)
print('REST      : POST ' + _public + '/ask   body {"question":"..."}')
print('            header  Authorization: Bearer ' + API_TOKEN)
print('Voice     : POST ' + _public + '/ask_voice   (multipart file=<wav>)')
print('=' * 66)
print()
print('Paste the URL + token into the frontend Settings panel → Save & connect.')
print('Leave this cell running. To restart the server after a code change,')
print('just rerun this cell — no model reload needed.')